In [ ]:
# ====================================================================
# 📅 CONFIGURAÇÃO DO ANO - Execute esta célula PRIMEIRO!
# ====================================================================

import pandas as pd
import os
from datetime import datetime
import shutil

print("="*70)
print("🎯 SISTEMA DE PROCESSAMENTO DE DADOS BUD - TC")
print("="*70)

# Input do ano
ano_padrao = datetime.now().year
ano_input = input(f"\n📅 Digite o ano para processar (padrão: {ano_padrao}): ")
ANO_ATUAL = int(ano_input) if ano_input.strip() else ano_padrao

print(f"\n✅ Ano selecionado: {ANO_ATUAL}")

# ====================================================================
# 📁 ESTRUTURA DE PASTAS
# ====================================================================

PASTA_MODULO = os.path.join('dados', 'TC_Ext')
PASTA_ANO = os.path.join(PASTA_MODULO, str(ANO_ATUAL))
PASTA_BUD = os.path.join(PASTA_ANO, 'BUD')  # Pasta específica para outputs de BUD
PASTA_HISTORICO = os.path.join(PASTA_MODULO, 'historico_consolidado')
PASTA_HISTORICO_BUD = os.path.join(PASTA_HISTORICO, 'BUD')  # Histórico específico para BUD
PASTA_RAIZ = '.'  # Onde estão os arquivos originais (raiz do projeto)

# Criar estrutura de pastas
os.makedirs(PASTA_ANO, exist_ok=True)
os.makedirs(PASTA_BUD, exist_ok=True)
os.makedirs(PASTA_HISTORICO, exist_ok=True)
os.makedirs(PASTA_HISTORICO_BUD, exist_ok=True)

print(f"\n📁 Estrutura de pastas criada:")
print(f"   ✅ {PASTA_ANO}/")
print(f"   ✅ {PASTA_BUD}/")
print(f"   ✅ {PASTA_HISTORICO}/")
print(f"   ✅ {PASTA_HISTORICO_BUD}/")

# ====================================================================
# 📋 VERIFICAR E ORGANIZAR ARQUIVOS DE ENTRADA
# ====================================================================

arquivos_necessarios = {
    'KE5Z_veiculos.xlsx': 'Dados de veículos KE5Z',
    'Dados SAPIENS.xlsx': 'Base de dados SAPIENS',
    'Reporting fluxo anexo.xlsx': 'Dados de rateio/volume'
}

print(f"\n🔍 Verificando arquivos de entrada...")

arquivos_ok = []
arquivos_faltando = []

for arquivo, descricao in arquivos_necessarios.items():
    caminho_ano = os.path.join(PASTA_ANO, arquivo)
    caminho_bud = os.path.join(PASTA_BUD, arquivo)
    caminho_raiz = os.path.join(PASTA_RAIZ, arquivo)
    
    # Verificar se arquivo existe na pasta do ano
    if os.path.exists(caminho_ano):
        arquivos_ok.append(arquivo)
        print(f"   ✅ {arquivo} - encontrado em {PASTA_ANO}/")
    
    # Verificar se arquivo existe na pasta BUD
    elif os.path.exists(caminho_bud):
        arquivos_ok.append(arquivo)
        print(f"   ✅ {arquivo} - encontrado em {PASTA_BUD}/")
    
    # Se não existe, verificar na raiz
    elif os.path.exists(caminho_raiz):
        print(f"   📋 {arquivo} - encontrado na raiz, copiando para {PASTA_ANO}/")
        shutil.copy2(caminho_raiz, caminho_ano)
        arquivos_ok.append(arquivo)
        print(f"   ✅ {arquivo} - copiado com sucesso!")
    
    else:
        arquivos_faltando.append((arquivo, descricao))
        print(f"   ❌ {arquivo} - NÃO ENCONTRADO")

# ====================================================================
# ⚠️ VALIDAÇÃO
# ====================================================================

if arquivos_faltando:
    print(f"\n{'='*70}")
    print(f"⚠️  ATENÇÃO: Arquivos não encontrados!")
    print(f"{'='*70}")
    for arquivo, descricao in arquivos_faltando:
        print(f"   ❌ {arquivo}")
        print(f"      Descrição: {descricao}")
        print(f"      Copie para: {PASTA_ANO}/{arquivo} ou {PASTA_BUD}/{arquivo}")
    print(f"{'='*70}")
    
    continuar = input(f"\n⚠️  Deseja continuar mesmo assim? (s/N): ")
    if continuar.lower() != 's':
        raise Exception("❌ Processamento cancelado - arquivos não encontrados")
    print(f"\n⚠️  Continuando com arquivos disponíveis...")

# ====================================================================
# 📊 DEFINIR CAMINHOS DE ENTRADA E SAÍDA
# ====================================================================

# Caminhos de entrada (priorizar pasta do ano, depois pasta BUD, depois raiz)
def encontrar_arquivo(nome_arquivo):
    """Encontra o arquivo na ordem: pasta do ano, pasta BUD, raiz"""
    caminho_ano = os.path.join(PASTA_ANO, nome_arquivo)
    caminho_bud = os.path.join(PASTA_BUD, nome_arquivo)
    caminho_raiz = os.path.join(PASTA_RAIZ, nome_arquivo)
    
    if os.path.exists(caminho_ano):
        return caminho_ano
    elif os.path.exists(caminho_bud):
        return caminho_bud
    elif os.path.exists(caminho_raiz):
        return caminho_raiz
    else:
        return caminho_ano  # Retorna caminho padrão mesmo se não existir

CAMINHO_KE5Z = encontrar_arquivo('KE5Z_veiculos.xlsx')
CAMINHO_SAPIENS = encontrar_arquivo('Dados SAPIENS.xlsx')
CAMINHO_RATEIO = 'Reporting fluxo anexo.xlsx'

# Caminhos de saída (parquets na pasta BUD com sufixo BUD)
CAMINHO_DF_FINAL = os.path.join(PASTA_BUD, 'df_final_BUD.parquet')
CAMINHO_DF_VOL = os.path.join(PASTA_BUD, 'df_vol_BUD.parquet')
CAMINHO_DF_KE5Z_GROUP = os.path.join(PASTA_BUD, 'df_ke5z_group_BUD.parquet')

# Caminhos de saída (Excel na pasta BUD com sufixo BUD)
CAMINHO_DF_FINAL_XLSX = os.path.join(PASTA_BUD, 'df_final_BUD.xlsx')
CAMINHO_DF_VOL_XLSX = os.path.join(PASTA_BUD, 'df_vol_BUD.xlsx')
CAMINHO_DF_KE5Z_GROUP_XLSX = os.path.join(PASTA_BUD, 'df_ke5z_group_BUD.xlsx')
CAMINHO_DF_FINAL_CPU_XLSX = os.path.join(PASTA_BUD, 'df_final_cpu_BUD.xlsx')

# Caminhos do histórico consolidado BUD
CAMINHO_HISTORICO_FINAL = os.path.join(PASTA_HISTORICO_BUD, 'df_final_historico_BUD.parquet')
CAMINHO_HISTORICO_VOL = os.path.join(PASTA_HISTORICO_BUD, 'df_vol_historico_BUD.parquet')
CAMINHO_HISTORICO_KE5Z = os.path.join(PASTA_HISTORICO_BUD, 'df_ke5z_historico_BUD.parquet')

# ====================================================================
# 📊 RESUMO DA CONFIGURAÇÃO
# ====================================================================

print(f"\n{'='*70}")
print(f"📊 RESUMO DA CONFIGURAÇÃO BUD")
print(f"{'='*70}")
print(f"📅 Ano: {ANO_ATUAL}")
print(f"\n📥 Arquivos de Entrada:")
for arquivo in arquivos_ok:
    caminho_entrada = os.path.join(PASTA_ANO, arquivo) if os.path.exists(os.path.join(PASTA_ANO, arquivo)) else os.path.join(PASTA_BUD, arquivo)
    print(f"   ✅ {caminho_entrada}")
print(f"\n💾 Arquivos de Saída (Parquet - BUD):")
print(f"   📄 {CAMINHO_DF_FINAL}")
print(f"   📄 {CAMINHO_DF_VOL}")
print(f"   📄 {CAMINHO_DF_KE5Z_GROUP}")
print(f"\n💾 Arquivos de Saída (Excel - BUD):")
print(f"   📄 {CAMINHO_DF_FINAL_XLSX}")
print(f"   📄 {CAMINHO_DF_VOL_XLSX}")
print(f"   📄 {CAMINHO_DF_KE5Z_GROUP_XLSX}")
print(f"   📄 {CAMINHO_DF_FINAL_CPU_XLSX}")
print(f"\n📚 Histórico Consolidado (BUD):")
print(f"   📄 {CAMINHO_HISTORICO_FINAL}")
print(f"   📄 {CAMINHO_HISTORICO_VOL}")
print(f"   📄 {CAMINHO_HISTORICO_KE5Z}")
print(f"{'='*70}")

confirmar = input(f"\n✅ Confirma o processamento BUD do ano {ANO_ATUAL}? (S/n): ")
if confirmar.lower() in ['n', 'nao', 'não']:
    raise Exception("❌ Processamento cancelado pelo usuário")

print(f"\n🚀 Configuração BUD confirmada!")
print(f"{'='*70}")
print(f"📝 Execute as próximas células para processar os dados BUD")
print(f"📁 Arquivos serão salvos em: {PASTA_BUD}/")
print(f"{'='*70}\n")


In [ ]:
# Ler o arquivo Reporting fluxo anexo.xlsx na guia "Voz de custo BDG"
# e transformar as colunas de meses em uma coluna Período
import pandas as pd

# Usar caminho configurado ou padrão
arquivo_rateio = CAMINHO_RATEIO if 'CAMINHO_RATEIO' in globals() else 'Reporting fluxo anexo.xlsx'

# Ler a guia "Voz de custo BDG" do arquivo Excel
df_KE5Z = pd.read_excel(arquivo_rateio, sheet_name='Voz de custo BDG')

# Identificar as colunas que são meses (janeiro a dezembro)
meses = ['janeiro', 'fevereiro', 'março', 'abril', 'maio', 'junho',
         'julho', 'agosto', 'setembro', 'outubro', 'novembro', 'dezembro']

# Encontrar as colunas que são meses (desconsiderando capitalização)
colunas_meses = [col for col in df_KE5Z.columns if any(mes.lower() in str(col).lower() for mes in meses)]

# Identificar as colunas que NÃO são meses (para usar como id_vars)
colunas_id = [col for col in df_KE5Z.columns if col not in colunas_meses]

# Transformar as colunas de meses em linhas usando melt
df_KE5Z = df_KE5Z.melt(
    id_vars=colunas_id,
    value_vars=colunas_meses,
    var_name='Período',
    value_name='Valor'
)

# Converter a coluna Valor para numérico, substituir NaN por zero
df_KE5Z['Valor'] = pd.to_numeric(df_KE5Z['Valor'], errors='coerce').fillna(0)

# 🔧 CORREÇÃO: Normalizar o nome do Período (capitalizar primeira letra) - MESMA LÓGICA DO dados.ipynb
# Mapear meses para formato capitalizado
mapeamento_meses = {
    'janeiro': 'Janeiro', 'fevereiro': 'Fevereiro', 'março': 'Março',
    'abril': 'Abril', 'maio': 'Maio', 'junho': 'Junho',
    'julho': 'Julho', 'agosto': 'Agosto', 'setembro': 'Setembro',
    'outubro': 'Outubro', 'novembro': 'Novembro', 'dezembro': 'Dezembro'
}

df_KE5Z['Período'] = df_KE5Z['Período'].astype(str).str.strip()
for mes_min, mes_cap in mapeamento_meses.items():
    df_KE5Z['Período'] = df_KE5Z['Período'].str.replace(mes_min, mes_cap, case=False, regex=False)

# Excluir as colunas Var/Fix e Ano se existirem
colunas_para_remover = []
if 'Var/Fix' in df_KE5Z.columns:
    colunas_para_remover.append('Var/Fix')
if 'Ano' in df_KE5Z.columns:
    colunas_para_remover.append('Ano')

if colunas_para_remover:
    df_KE5Z = df_KE5Z.drop(columns=colunas_para_remover)
    print(f"✅ Colunas removidas: {', '.join(colunas_para_remover)}")

# mostrar o arquivo em excel df_KE5Z
df_KE5Z.head(50)

# Fazer o somatorio da coluna Valor e imprimir na tela
print(f"Soma da coluna Valor: {df_KE5Z['Valor'].sum()}")

# Remover a coluna Custo do df_KE5Z se existir
if 'Custo' in df_KE5Z.columns:
    df_KE5Z = df_KE5Z.drop(columns=['Custo'])
    print("✅ Coluna 'Custo' removida do df_KE5Z")

# exibir as primeiras linhas
df_KE5Z.head(20)

















In [ ]:
# Ler guia "Base conso" e garantir só uma coluna 'Custo' no resultado (BUD)
# Usar caminho configurado ou padrão
arquivo_sapiens = CAMINHO_SAPIENS if 'CAMINHO_SAPIENS' in globals() else 'Dados SAPIENS.xlsx'

# Verificar se o arquivo existe antes de ler
import os
if not os.path.exists(arquivo_sapiens):
    print(f"⚠️ AVISO: Arquivo SAPIENS não encontrado em: {arquivo_sapiens}")
    print("⚠️ Continuando sem merge de Custo...")
    df_base_conso = pd.DataFrame()
else:
    df_base_conso = pd.read_excel(arquivo_sapiens, sheet_name='Base conso')
    
    # Renomear Type 04 para Custo se existir no Excel
    if 'Type 04' in df_base_conso.columns:
        df_base_conso = df_base_conso.rename(columns={'Type 04': 'Custo'})
    
    # Verificar se as colunas necessárias existem
    if 'Custo' in df_base_conso.columns and 'Type 07' in df_base_conso.columns:
        # Manter somente a coluna Custo e Type 07
        df_base_conso = df_base_conso[['Custo', 'Type 07']]
        
        # Renomear a coluna Type 07 para Account
        df_base_conso = df_base_conso.rename(columns={'Type 07': 'Account'})
        
        # Remover duplicatas de Account (manter apenas o primeiro)
        df_base_conso = df_base_conso.drop_duplicates(subset=['Account'], keep='first')
        
        # mostrar as primeiras linhas
        print("📊 Primeiras linhas do df_base_conso (BUD):")
        display(df_base_conso.head(30))
        
        # Verificar se df_KE5Z tem a coluna Account antes do merge
        if 'Account' in df_KE5Z.columns:
            # fazer merge utilizando a coluna Account como chave e retornar a Custo para o df_KE5Z
            df_KE5Z = pd.merge(df_KE5Z, df_base_conso[['Custo', 'Account']], on='Account', how='left')
            print(f"✅ Merge com SAPIENS concluído! Coluna 'Custo' adicionada ao df_KE5Z (BUD)")
        else:
            print("⚠️ AVISO: Coluna 'Account' não encontrada em df_KE5Z. Merge não realizado.")
    else:
        print("⚠️ AVISO: Colunas 'Custo' ou 'Type 07' não encontradas no arquivo SAPIENS.")
        print("⚠️ Continuando sem merge de Custo...")

# mostrar as primeiras linhas do df_KE5Z após o merge
print("\n📊 Primeiras linhas do df_KE5Z após merge (BUD):")
display(df_KE5Z.head(30))













In [ ]:
# Ler o arquivo em excel Reporting fluxo anexo.xlsx, ler a guia Rateio BDG,
# excluir a primeira linha (linha de referência) e usar a segunda linha como cabeçalho (meses)

# Usar caminho configurado ou padrão
arquivo_rateio = CAMINHO_RATEIO if 'CAMINHO_RATEIO' in globals() else 'Reporting fluxo anexo.xlsx'

# Ler a guia "Rateio BDG" do arquivo Excel, sem header para manipular manualmente
try:
    df_raw = pd.read_excel(arquivo_rateio, sheet_name='Rateio BDG', header=None)
except ValueError as e:
    if "Worksheet named 'Rateio BDG' not found" in str(e):
        print(f"⚠️ ERRO: Guia 'Rateio BDG' não encontrada no arquivo: {arquivo_rateio}")
        # Listar todas as guias disponíveis
        xl_file = pd.ExcelFile(arquivo_rateio)
        print(f"📋 Guias disponíveis no arquivo:")
        for sheet in xl_file.sheet_names:
            print(f"   - {sheet}")
        # Tentar encontrar guia similar
        guias_similares = [s for s in xl_file.sheet_names if 'rateio' in s.lower() or 'bdg' in s.lower()]
        if guias_similares:
            print(f"\n💡 Guias similares encontradas: {guias_similares}")
            print(f"   Tentando usar: {guias_similares[0]}")
            df_raw = pd.read_excel(arquivo_rateio, sheet_name=guias_similares[0], header=None)
        else:
            raise ValueError(f"Guia 'Rateio BDG' não encontrada e nenhuma guia similar foi encontrada.")
    else:
        raise

# Excluir a primeira linha (linha de referência)
df = df_raw.iloc[1:].reset_index(drop=True)

# Usar a primeira linha (que agora é a linha dos nomes/meses) como cabeçalho real
df.columns = df.iloc[0]

# Excluir a linha usada como cabeçalho
df = df.iloc[1:].reset_index(drop=True)

# Remover colunas totalmente NaN (colunas extras do Excel)
df = df.loc[:, df.notna().any(axis=0)]

# Filtrar colunas que possuem todos os valores NaN (antes do melt)
df = df.dropna(axis=1, how='all')

# Identificar as colunas que são meses (janeiro a dezembro)
meses = ['Janeiro', 'Fevereiro', 'Março', 'Abril', 'Maio', 'Junho',
         'Julho', 'Agosto', 'Setembro', 'Outubro', 'Novembro', 'Dezembro']

# Encontrar as colunas que são meses (desconsiderando capitalização)
colunas_meses = [col for col in df.columns if any(mes.lower() in str(col).lower() for mes in meses)]

# Identificar as colunas que NÃO são meses (para usar como id_vars)
colunas_id = [col for col in df.columns if col not in colunas_meses and pd.notna(col)]

# Remover colunas com nome NaN
df = df.loc[:, df.columns.notna()]



# Agora transformar as colunas de meses em linhas
df = df.melt(id_vars=colunas_id, value_vars=colunas_meses, var_name='Mês', value_name='Rateio')

# Converter a coluna Rateio para numérico, substituir NaN por zero
# NÃO arredondar para manter máxima precisão e evitar erros de arredondamento
df['Rateio'] = pd.to_numeric(df['Rateio'], errors='coerce').fillna(0)

# substituir o nome da coluna Mês por Período
df = df.rename(columns={'Mês': 'Período'})

# 🔧 CORREÇÃO: Normalizar o Período para capitalizar primeira letra (MESMA LÓGICA DO dados.ipynb)
# Mapear meses para formato capitalizado
mapeamento_meses = {
    'janeiro': 'Janeiro', 'fevereiro': 'Fevereiro', 'março': 'Março',
    'abril': 'Abril', 'maio': 'Maio', 'junho': 'Junho',
    'julho': 'Julho', 'agosto': 'Agosto', 'setembro': 'Setembro',
    'outubro': 'Outubro', 'novembro': 'Novembro', 'dezembro': 'Dezembro'
}

df['Período'] = df['Período'].astype(str).str.strip()
for mes_min, mes_cap in mapeamento_meses.items():
    df['Período'] = df['Período'].str.replace(mes_min, mes_cap, case=False, regex=False)

# filtrar na tabela df e linha Oficina tudo que é diferente de veículo
df = df[df['Oficina'] != 'Veículos']

# tirar o nan no filtro
df = df[df['Oficina'].notna()]



# mostrar um somatorio da coluna Rateio
print(df['Rateio'].sum())
# exibir as primeiras linhas
df.head(100)







In [ ]:
# --- VERIFICAÇÃO DE ERROS E USO DAS CHAVES 'Oficina' e 'Período' ---

# 1. Conferir colunas presentes
print("Colunas em df_KE5Z:", df_KE5Z.columns.tolist())
print("Colunas em df    :", df.columns.tolist())

# 2. Checar existência das colunas essenciais
erros = []
for nome_df, dfx in [('df_KE5Z', df_KE5Z), ('df', df)]:
    for col in ['Oficina', 'Período']:
        if col not in dfx.columns:
            erros.append(f"Coluna '{col}' ausente no {nome_df}.")

if erros:
    for erro in erros:
        print("ERRO:", erro)
    raise KeyError(" ".join(erros))

# 3. NORMALIZAR valores antes de qualquer verificação
print("\n" + "="*70)
print("🔧 NORMALIZANDO valores antes do merge")
print("="*70)

# 🔧 CORREÇÃO: Normalizar Período para capitalizar primeira letra (MESMA LÓGICA DO dados.ipynb)
# Mapear meses para formato capitalizado
mapeamento_meses = {
    'janeiro': 'Janeiro', 'fevereiro': 'Fevereiro', 'março': 'Março',
    'abril': 'Abril', 'maio': 'Maio', 'junho': 'Junho',
    'julho': 'Julho', 'agosto': 'Agosto', 'setembro': 'Setembro',
    'outubro': 'Outubro', 'novembro': 'Novembro', 'dezembro': 'Dezembro'
}

# Normalizar Período: remover espaços e capitalizar primeira letra
df_KE5Z['Período'] = df_KE5Z['Período'].astype(str).str.strip()
for mes_min, mes_cap in mapeamento_meses.items():
    df_KE5Z['Período'] = df_KE5Z['Período'].str.replace(mes_min, mes_cap, case=False, regex=False)

df['Período'] = df['Período'].astype(str).str.strip()
for mes_min, mes_cap in mapeamento_meses.items():
    df['Período'] = df['Período'].str.replace(mes_min, mes_cap, case=False, regex=False)

# Normalizar Oficina: remover espaços
df_KE5Z['Oficina'] = df_KE5Z['Oficina'].astype(str).str.strip()
df['Oficina'] = df['Oficina'].astype(str).str.strip()

print("✅ Valores normalizados (strip + capitalize para Período, strip para Oficina)")

# Verificar correspondências após normalização
periodos_ke5z = set(df_KE5Z['Período'].unique())
periodos_df = set(df['Período'].unique())
periodos_comum = periodos_ke5z.intersection(periodos_df)

print(f"\n📊 ANÁLISE DE PERÍODOS:")
print(f"   Períodos em df_KE5Z: {sorted(periodos_ke5z)}")
print(f"   Períodos em df     : {sorted(periodos_df)}")
print(f"   ✅ Períodos em comum: {sorted(periodos_comum)}")
if periodos_ke5z - periodos_df:
    print(f"   ⚠️ Períodos apenas em df_KE5Z: {sorted(periodos_ke5z - periodos_df)}")
if periodos_df - periodos_ke5z:
    print(f"   ⚠️ Períodos apenas em df: {sorted(periodos_df - periodos_ke5z)}")

oficinas_ke5z = set(df_KE5Z['Oficina'].unique())
oficinas_df = set(df['Oficina'].unique())
oficinas_comum = oficinas_ke5z.intersection(oficinas_df)

print(f"\n📊 ANÁLISE DE OFICINAS:")
print(f"   Total de oficinas em df_KE5Z: {len(oficinas_ke5z)}")
print(f"   Total de oficinas em df     : {len(oficinas_df)}")
print(f"   ✅ Oficinas em comum: {len(oficinas_comum)}")
if len(oficinas_comum) < 5:
    print(f"   ⚠️ Oficinas em comum (primeiras 20): {sorted(list(oficinas_comum))[:20]}")
if oficinas_ke5z - oficinas_df:
    print(f"   ⚠️ Oficinas apenas em df_KE5Z (primeiras 10): {sorted(list(oficinas_ke5z - oficinas_df))[:10]}")
if oficinas_df - oficinas_ke5z:
    print(f"   ⚠️ Oficinas apenas em df (primeiras 10): {sorted(list(oficinas_df - oficinas_ke5z))[:10]}")

# Verificar combinações únicas
print(f"\n📊 ANÁLISE DE COMBINAÇÕES:")
combinacoes_ke5z = set(zip(df_KE5Z['Oficina'], df_KE5Z['Período']))
combinacoes_df = set(zip(df['Oficina'], df['Período']))
combinacoes_comum = combinacoes_ke5z.intersection(combinacoes_df)

print(f"   Combinações únicas em df_KE5Z: {len(combinacoes_ke5z)}")
print(f"   Combinações únicas em df     : {len(combinacoes_df)}")
print(f"   ✅ Combinações em comum: {len(combinacoes_comum)}")
if len(combinacoes_comum) > 0 and len(combinacoes_comum) <= 10:
    print(f"   Exemplos de combinações em comum: {sorted(list(combinacoes_comum))[:10]}")

print("="*70 + "\n")

# 4. Conferir quantidade de linhas antes do merge
print(f"Usando 'Oficina' e 'Período' como chaves de merge")
print(f"Linhas em df_KE5Z: {len(df_KE5Z)}")
print(f"Linhas em df     : {len(df)}")

# 5. Realizar merge tendo certeza do nome correto das chaves
try:
    df_merge = pd.merge(df_KE5Z, df, on=['Oficina', 'Período'], how='left', suffixes=('', '_df'))
except Exception as e:
    print("Erro ao realizar o merge usando as colunas 'Oficina' e 'Período'.")
    raise

print(f"Linhas após merge: {len(df_merge)}")

# 6. Checar se a coluna Rateio veio corretamente
if 'Rateio' in df_merge.columns:
    rateio_nao_nulo = df_merge['Rateio'].notna().sum()
    print(f"Linhas com Rateio encontrado: {rateio_nao_nulo}")
else:
    print("AVISO: Coluna 'Rateio' não encontrada após o merge.")

# 7. Validar presença da coluna 'Veículo' para o pivot
if 'Veículo' not in df_merge.columns:
    raise KeyError("Coluna 'Veículo' não encontrada em df_merge. Verifique se esta coluna existe e está corretamente capitalizada em ambos DataFrames.")

# 8. Pivot para transformar veículos em colunas de Rateio
# Usar 'mean' para agregar valores duplicados (mais apropriado para rateios)
try:
    df_pivot = df_merge.pivot_table(
        index=['Oficina', 'Período'],
        columns='Veículo',
        values='Rateio',
        aggfunc='mean'
    ).reset_index()
    df_pivot.columns.name = None
    print(f"\nLinhas após pivot (Oficina + Período): {len(df_pivot)}")
except Exception as e:
    print("Erro ao executar pivot_table em df_merge usando 'Veículo'.")
    raise

# 9. Merge reverso: incluir dados originais do KE5Z
try:
    df_final = pd.merge(df_KE5Z, df_pivot, on=['Oficina', 'Período'], how='left')
except Exception as e:
    print("Erro ao executar o merge final com pivot.")
    raise

# 10. Determinar colunas novas (veículos criados) e renomear para %
veiculos_cols = [col for col in df_final.columns if col not in df_KE5Z.columns and col not in ['Oficina', 'Período']]
rename_dict = {col: f"{col}%" for col in veiculos_cols}
df_final = df_final.rename(columns=rename_dict)

# Atualizar lista de veiculos_cols para aquelas com %
veiculos_cols_pct = [f"{col}%" for col in veiculos_cols]
veiculos_cols = [col for col in veiculos_cols_pct if col in df_final.columns]

# 11. Garantir que todas as colunas de veículos estejam em float64, sem perder casas decimais, e NaN->0
import numpy as np
for col in veiculos_cols:
    # Remover eventualmente o símbolo % para padronizar antes da conversão, se vier por engano
    if df_final[col].dtype == "object":
        df_final[col] = df_final[col].astype(str).str.replace('%', '', regex=False).str.strip()
    # Converter para float64 (não arredonda nem corta casas decimais)
    df_final[col] = pd.to_numeric(df_final[col], errors='coerce').astype(np.float64).fillna(0.0)

# 11. Diagnóstico final
print(f"\nDataFrame final criado com {len(df_final)} linhas e {len(df_final.columns)} colunas")
print(f"Colunas de veículos criadas: {len(veiculos_cols)}")
print("Colunas de veículos:", veiculos_cols)
print("\nTipos das colunas de veículos:")
for col in veiculos_cols:
    print(f"  {col}: {df_final[col].dtype} (exemplo valor: {df_final[col].dropna().iloc[0] if not df_final[col].dropna().empty else 'N/A'})")

# 12. Conferir dados de HVAC
if 'Oficina' in df_final.columns:
    df_hvac = df_final[df_final['Oficina'] == 'HVAC']
    print(f"\nLinhas com HVAC: {len(df_hvac)}")
   
else:
    print("AVISO: Coluna 'Oficina' não existe em df_final.")
    df_hvac = pd.DataFrame()




In [ ]:
# Criar novas colunas calculando: Coluna% * Valor
# As colunas de percentual estão como float (ex: 0.419 para 41.9%)
# Multiplicar diretamente pela coluna Valor

print("Criando colunas de cálculo (Percentual * Valor)...")

# Verificar se a coluna 'Valor' existe
if 'Valor' not in df_final.columns:
    print("ERRO: Coluna 'Valor' não encontrada no DataFrame!")
    print(f"Colunas disponíveis: {df_final.columns.tolist()}")
else:
    print(f"Coluna 'Valor' encontrada. Tipo: {df_final['Valor'].dtype}")
    
    # Converter coluna Valor para numérico se necessário
    df_final['Valor'] = pd.to_numeric(df_final['Valor'], errors='coerce').fillna(0)
    
    # Lista de colunas de veículos com %
    veiculos_cols_pct = ['CC21%', 'CC22%', 'CC24%', 'CC24 5L%', 'CC24 7L%', 'J516%']
    
    # Criar uma coluna para cada veículo com o cálculo
    for col_pct in veiculos_cols_pct:
        if col_pct in df_final.columns:
            # Nome da nova coluna sem o "%" (ex: "CC21")
            col_nome = col_pct.replace('%', '')
            
            # Calcular: Percentual * Valor
            # Como o percentual já está em decimal (ex: 0.419), multiplicar diretamente
            df_final[col_nome] = df_final[col_pct] * df_final['Valor']
            
            print(f"  Criada coluna '{col_nome}' = {col_pct} * Valor")
        else:
            print(f"  AVISO: Coluna '{col_pct}' não encontrada")
    
    print(f"\nTotal de colunas no DataFrame final: {len(df_final.columns)}")
    print(f"Novas colunas criadas: {[col.replace('%', '') for col in veiculos_cols_pct if col in df_final.columns]}")
    
    # Exibir as primeiras linhas para verificar
    print("\nPrimeiras linhas do DataFrame final:")
    display(df_final.head(10))

    # fazer somatorio da coluna Valor
    print(df_final['Valor'].sum())

    # somar as colunas CC21, CC22, CC24, CC24 5L, CC24 7L, J516
    print(df_final['CC21'].sum() + df_final['CC22'].sum() + df_final['CC24'].sum() + df_final['CC24 5L'].sum() + df_final['CC24 7L'].sum() + df_final['J516'].sum())

# gerar um excel com o df_final
arquivo_excel_final = CAMINHO_DF_FINAL_XLSX if 'CAMINHO_DF_FINAL_XLSX' in globals() else os.path.join(PASTA_BUD, 'df_final_BUD.xlsx')
df_final.to_excel(arquivo_excel_final, index=False)









In [ ]:
# ANÁLISE: Verificar se a soma dos percentuais está dando 100% em cada linha
print("="*70)
print("ANÁLISE: SOMA DOS PERCENTUAIS POR LINHA")
print("="*70)

# Lista de colunas de percentual
veiculos_cols_pct = ['CC21%', 'CC22%', 'CC24%', 'CC24 5L%', 'CC24 7L%', 'J516%']

# Calcular a soma dos percentuais para cada linha
df_final['Soma_Percentuais'] = df_final[veiculos_cols_pct].sum(axis=1)

# Verificar quantas linhas têm rateios (soma > 0)
linhas_com_rateio = (df_final['Soma_Percentuais'] > 0).sum()
linhas_sem_rateio = (df_final['Soma_Percentuais'] == 0).sum()

print(f"\n1. DISTRIBUIÇÃO DE LINHAS:")
print(f"   Linhas COM rateios (soma > 0): {linhas_com_rateio:,}")
print(f"   Linhas SEM rateios (soma = 0): {linhas_sem_rateio:,}")
print(f"   Total de linhas: {len(df_final):,}")

# Verificar se a soma está próxima de 1.0 (100%) nas linhas com rateio
df_com_rateio = df_final[df_final['Soma_Percentuais'] > 0]
if len(df_com_rateio) > 0:
    print(f"\n2. ANÁLISE DAS LINHAS COM RATEIOS:")
    print(f"   Soma média dos percentuais: {df_com_rateio['Soma_Percentuais'].mean():.4f}")
    print(f"   Soma mínima: {df_com_rateio['Soma_Percentuais'].min():.4f}")
    print(f"   Soma máxima: {df_com_rateio['Soma_Percentuais'].max():.4f}")
    
    # Contar linhas onde a soma não está próxima de 1.0
    linhas_fora_100 = df_com_rateio[abs(df_com_rateio['Soma_Percentuais'] - 1.0) > 0.01]
    print(f"\n   ⚠️ Linhas onde soma ≠ 100% (diferença > 1%): {len(linhas_fora_100)}")
    
    if len(linhas_fora_100) > 0:
        print(f"\n   Exemplos de linhas com soma diferente de 100%:")
        display(linhas_fora_100[['Oficina', 'Período', 'Valor', 'Soma_Percentuais'] + veiculos_cols_pct].head(10))

# Verificar especificamente julho e agosto
print(f"\n3. ANÁLISE ESPECÍFICA: JULHO E AGOSTO")
df_jul_ago = df_final[df_final['Período'].isin(['Julho', 'Agosto', 'julho', 'agosto'])]
if len(df_jul_ago) > 0:
    print(f"   Total de linhas: {len(df_jul_ago):,}")
    linhas_com_rateio_jul_ago = (df_jul_ago['Soma_Percentuais'] > 0).sum()
    print(f"   Linhas COM rateios: {linhas_com_rateio_jul_ago:,}")
    
    if linhas_com_rateio_jul_ago > 0:
        df_jul_ago_com_rateio = df_jul_ago[df_jul_ago['Soma_Percentuais'] > 0]
        print(f"   Soma média dos percentuais: {df_jul_ago_com_rateio['Soma_Percentuais'].mean():.4f}")
        
        linhas_fora_100_jul_ago = df_jul_ago_com_rateio[abs(df_jul_ago_com_rateio['Soma_Percentuais'] - 1.0) > 0.01]
        print(f"   ⚠️ Linhas onde soma ≠ 100%: {len(linhas_fora_100_jul_ago)}")
        
        if len(linhas_fora_100_jul_ago) > 0:
            print(f"\n   Exemplos de linhas problemáticas em Julho/Agosto:")
            display(linhas_fora_100_jul_ago[['Oficina', 'Período', 'Valor', 'Soma_Percentuais'] + veiculos_cols_pct].head(10))

# Verificar totais
print(f"\n4. VERIFICAÇÃO DE TOTAIS:")
soma_valor_total = df_final['Valor'].sum()
soma_valor_com_rateio = df_com_rateio['Valor'].sum() if len(df_com_rateio) > 0 else 0
soma_calc_total = df_final[['CC21', 'CC22', 'CC24', 'CC24 5L', 'CC24 7L', 'J516']].sum().sum()

print(f"   Soma total da coluna Valor: {soma_valor_total:,.2f}")
print(f"   Soma da coluna Valor (apenas linhas com rateio): {soma_valor_com_rateio:,.2f}")
print(f"   Soma das colunas calculadas: {soma_calc_total:,.2f}")
print(f"   Diferença: {soma_valor_total - soma_calc_total:,.2f}")
print(f"   Percentual coberto: {(soma_calc_total / soma_valor_total * 100):.2f}%")

print("\n" + "="*70)


In [ ]:
# Calcular a somatória de cada coluna (CC21, CC22, CC24, CC24 5L, CC24 7L, J516)

print("="*60)
print("SOMATÓRIA DE CADA COLUNA")
print("="*60)

# Lista de colunas para somar
colunas_para_somar = ['CC21', 'CC22', 'CC24', 'CC24 5L', 'CC24 7L', 'J516']

# Calcular e exibir a soma de cada coluna
soma_total = 0
for col in colunas_para_somar:
    if col in df_final.columns:
        # Converter para numérico e somar
        soma = pd.to_numeric(df_final[col], errors='coerce').fillna(0).sum()
        soma_total += soma
        print(f"Soma da coluna {col:12s}: {soma:,.2f}")
    else:
        print(f"Coluna {col:12s}: NÃO ENCONTRADA")

print("="*60)
print(f"SOMA TOTAL:                 {soma_total:,.2f}")
print("="*60)

#


In [ ]:
# Apagar as colunas de percentual uma a uma
print("Removendo colunas de percentual...")
colunas_para_remover = ['CC21%', 'CC22%', 'CC24%', 'CC24 5L%', 'CC24 7L%', 'J516%']

for col in colunas_para_remover:
    if col in df_final.columns:
        df_final = df_final.drop(columns=[col])
        print(f"  Coluna '{col}' removida")
    else:
        print(f"  AVISO: Coluna '{col}' não encontrada")

print(f"\nTotal de colunas após remoção: {len(df_final.columns)}")
print(f"Colunas restantes: {df_final.columns.tolist()}")



# mostrar o df_final_usi_tc_ext
display(df_final)




In [ ]:
# Transformar as colunas CC21, CC22, CC24, CC24 5L, CC24 7L, J516 em linhas, mantendo todas as outras colunas
colunas_veiculos = ['CC21', 'CC22', 'CC24', 'CC24 5L', 'CC24 7L', 'J516']
colunas_veiculos_existentes = [col for col in colunas_veiculos if col in df_final.columns]

if len(colunas_veiculos_existentes) > 0:
    colunas_id = [col for col in df_final.columns if col not in colunas_veiculos]
    df_final = df_final.melt(id_vars=colunas_id, value_vars=colunas_veiculos_existentes, var_name='Veículo', value_name='Total')
else:
    print("AVISO: Nenhuma das colunas de veículos foi encontrada!")
    print(f"Colunas disponíveis: {df_final.columns.tolist()}")

# mostrar o df_final
display(df_final)

# somar a coluna Total
print(df_final['Total'].sum())




In [ ]:
# ler o arquivo em excel Reporting fluxo anexo.xlsx na guia Volume BDG e considerar a linha 51 como cabeçalho
# Usar caminho configurado ou padrão
arquivo_rateio_vol = CAMINHO_RATEIO if 'CAMINHO_RATEIO' in globals() else 'Reporting fluxo anexo.xlsx'

# Ler o arquivo Excel na guia 'Volume BDG', considerando a linha 51 como cabeçalho (header=50, 0-indexed)
try:
    df_ke5z_volume = pd.read_excel(arquivo_rateio_vol, sheet_name='Volume BDG', header=50)
except ValueError as e:
    if "Worksheet named 'Volume BDG' not found" in str(e):
        print(f"⚠️ ERRO: Guia 'Volume BDG' não encontrada no arquivo: {arquivo_rateio_vol}")
        # Listar todas as guias disponíveis
        xl_file = pd.ExcelFile(arquivo_rateio_vol)
        print(f"📋 Guias disponíveis no arquivo:")
        for sheet in xl_file.sheet_names:
            print(f"   - {sheet}")
        # Tentar encontrar guia similar
        guias_similares = [s for s in xl_file.sheet_names if 'volume' in s.lower() or 'bdg' in s.lower()]
        if guias_similares:
            print(f"\n💡 Guias similares encontradas: {guias_similares}")
            print(f"   Tentando usar: {guias_similares[0]}")
            df_ke5z_volume = pd.read_excel(arquivo_rateio_vol, sheet_name=guias_similares[0], header=50)
        else:
            raise ValueError(f"Guia 'Volume BDG' não encontrada e nenhuma guia similar foi encontrada.")
    else:
        raise


# Excluir colunas sem informações (todas as entradas NaN) ou sem títulos (coluna com nome NaN ou string vazia)
df_ke5z_volume = df_ke5z_volume.dropna(axis=1, how='all')
df_ke5z_volume = df_ke5z_volume.loc[:, [col for col in df_ke5z_volume.columns if (not (pd.isna(col) or str(col).strip() == ""))]]


# 🔧 CORREÇÃO: Normalizar nomes dos meses para corresponder ao formato usado nas outras células
# Mapear meses em minúsculas para capitalizados (primeira letra maiúscula) - MESMA LÓGICA DO dados.ipynb
mapeamento_meses = {
    'janeiro': 'Janeiro',
    'fevereiro': 'Fevereiro',
    'março': 'Março',
    'abril': 'Abril',
    'maio': 'Maio',
    'junho': 'Junho',
    'julho': 'Julho',
    'agosto': 'Agosto',
    'setembro': 'Setembro',
    'outubro': 'Outubro',
    'novembro': 'Novembro',
    'dezembro': 'Dezembro'
}

# Identificar colunas de meses (pode estar em minúsculas ou capitalizadas no Excel)
colunas_meses_minusculas = ['janeiro', 'fevereiro', 'março', 'abril', 'maio', 'junho', 'julho', 'agosto', 'setembro', 'outubro', 'novembro', 'dezembro']
colunas_meses_capitalizadas = ['Janeiro', 'Fevereiro', 'Março', 'Abril', 'Maio', 'Junho', 'Julho', 'Agosto', 'Setembro', 'Outubro', 'Novembro', 'Dezembro']

# Encontrar as colunas que são meses no DataFrame (pode estar em qualquer formato)
colunas_meses_encontradas = []
for col in df_ke5z_volume.columns:
    col_lower = str(col).lower().strip()
    if col_lower in [m.lower() for m in colunas_meses_minusculas]:
        colunas_meses_encontradas.append(col)

# Se não encontrou, tentar formato capitalizado
if not colunas_meses_encontradas:
    for col in df_ke5z_volume.columns:
        if str(col).strip() in colunas_meses_capitalizadas:
            colunas_meses_encontradas.append(col)

print(f"📊 Colunas de meses encontradas: {colunas_meses_encontradas}")

# Derreter o DataFrame para transformar as colunas em linhas
df_vol = pd.melt(
    df_ke5z_volume,
    id_vars=[col for col in df_ke5z_volume.columns if col not in colunas_meses_encontradas],
    value_vars=colunas_meses_encontradas,
    var_name='Período',
    value_name='Volume'
)

# 🔧 CORREÇÃO CRÍTICA: Normalizar os nomes dos períodos para capitalizar primeira letra
# Isso garante que o merge na célula 12 funcione corretamente
df_vol['Período'] = df_vol['Período'].astype(str).str.strip()
for mes_min, mes_cap in mapeamento_meses.items():
    df_vol['Período'] = df_vol['Período'].str.replace(mes_min, mes_cap, case=False, regex=False)

# Transformar a coluna Volume em numerico
df_vol['Volume'] = pd.to_numeric(df_vol['Volume'], errors='coerce').fillna(0)


# Remover linhas duplicadas
df_vol = df_vol.drop_duplicates()  


# Remover linhas com NaN
df_vol = df_vol.dropna()

# Exibir as primeiras linhas para conferência
display(df_vol)

# gerar um arquivo parquet com o df_vol
arquivo_parquet_vol = CAMINHO_DF_VOL if 'CAMINHO_DF_VOL' in globals() else os.path.join(PASTA_BUD, 'df_vol_BUD.parquet')
df_vol.to_parquet(arquivo_parquet_vol)



In [ ]:

# Removido: merge de Volume e cálculo de CPU - agora calculados diretamente nos arquivos Streamlit (TC_Ext.py e app.py)

# filtrar account diferente de NaN, 0 ou TC Ext
df_final = df_final[df_final['Account'].notna() & (df_final['Account'] != 0) & (df_final['Account'] != 'TC Ext')]

# Gerar excel com o df_final
arquivo_excel_cpu = CAMINHO_DF_FINAL_CPU_XLSX if 'CAMINHO_DF_FINAL_CPU_XLSX' in globals() else os.path.join(PASTA_BUD, 'df_final_cpu_BUD.xlsx')
df_final.to_excel(arquivo_excel_cpu, index=False)



# A coluna já vem como 'Custo' desde o merge na célula 1, não precisa renomear

# mostrar o df_final
display(df_final)



# somar a coluna Valor e coluna Total
print('Valor = ', df_final['Valor'].sum())
print('Total = ', df_final['Total'].sum())

# Gerar arquivo parquet com o df_final
arquivo_parquet_final = CAMINHO_DF_FINAL if 'CAMINHO_DF_FINAL' in globals() else os.path.join(PASTA_BUD, 'df_final_BUD.parquet')
df_final.to_parquet(arquivo_parquet_final)







In [ ]:

# 3. Agrupar Volume
try:
    df_vol_group = (
        df_vol.groupby(['Oficina', 'Período'], as_index=False)['Volume']
        .sum()
    )
except Exception as e:
    print("ERRO ao agrupar df_vol:", e)
    raise

# 🔧 VALIDAÇÃO: Verificar formato dos períodos antes do merge
print(f"\n📊 Validação antes do merge:")
print(f"   Períodos únicos em df_vol_group: {sorted(df_vol_group['Período'].unique())}")
print(f"   Total de Volume agrupado: {df_vol_group['Volume'].sum():,.0f}")
print(f"   Linhas em df_vol_group: {len(df_vol_group):,}")

# Exibir as primeiras 100 linhas
display(df_vol_group.head(100))



# 5. Garantir coluna 'Account' presente e aplicar filtros
if 'Account' not in df_KE5Z.columns:
    raise ValueError("df_KE5Z não contém coluna 'Account'")
df_KE5Z = df_KE5Z[df_KE5Z['Account'].notna() & (df_KE5Z['Account'] != 0) & (df_KE5Z['Account'] != '')]

# 6. Checar se as colunas 'Volume' e 'Total' (ou 'Valor') existem e são numéricas
# Volume
if 'Volume' not in df_KE5Z.columns:
    print("Coluna 'Volume' não encontrada em df_KE5Z, criando com zeros.")
    df_KE5Z['Volume'] = 0

df_KE5Z['Volume'] = pd.to_numeric(df_KE5Z['Volume'], errors='coerce').fillna(0)

# Total
if 'Total' not in df_KE5Z.columns:
    if 'Valor' in df_KE5Z.columns:
        df_KE5Z['Total'] = df_KE5Z['Valor']
        print("Coluna 'Total' criada a partir de 'Valor' em df_KE5Z.")
    else:
        print("Coluna 'Total' e 'Valor' não encontradas em df_KE5Z, criando 'Total' com zeros.")
        df_KE5Z['Total'] = 0

df_KE5Z['Total'] = pd.to_numeric(df_KE5Z['Total'], errors='coerce').fillna(0)

# 🔧 VALIDAÇÃO: Verificar períodos em df_KE5Z antes do merge
if 'Período' in df_KE5Z.columns:
    print(f"\n📊 Períodos únicos em df_KE5Z: {sorted(df_KE5Z['Período'].unique())}")
    print(f"   Linhas em df_KE5Z: {len(df_KE5Z):,}")

# 🔧 CORREÇÃO: Normalizar períodos em df_KE5Z para garantir correspondência
if 'Período' in df_KE5Z.columns:
    # Mapear meses para formato capitalizado (mesma lógica da célula 10)
    mapeamento_meses = {
        'janeiro': 'Janeiro', 'fevereiro': 'Fevereiro', 'março': 'Março',
        'abril': 'Abril', 'maio': 'Maio', 'junho': 'Junho',
        'julho': 'Julho', 'agosto': 'Agosto', 'setembro': 'Setembro',
        'outubro': 'Outubro', 'novembro': 'Novembro', 'dezembro': 'Dezembro'
    }
    df_KE5Z['Período'] = df_KE5Z['Período'].astype(str).str.strip()
    for mes_min, mes_cap in mapeamento_meses.items():
        df_KE5Z['Período'] = df_KE5Z['Período'].str.replace(mes_min, mes_cap, case=False, regex=False)

# Fazer o merge entre df_KE5Z e df_vol_group pela chave Oficina e Período
# Garante que só haverá uma coluna 'Volume' ao final
df_ke5z_group = pd.merge(
    df_KE5Z.drop(columns=[col for col in df_KE5Z.columns if col.lower() == 'volume']),  # remove 'Volume' antes!
    df_vol_group,
    on=['Oficina', 'Período'],
    how='left'
)

# 🔧 VALIDAÇÃO: Verificar resultado do merge
print(f"\n📊 Validação após merge:")
print(f"   Linhas após merge: {len(df_ke5z_group):,}")
if 'Volume' in df_ke5z_group.columns:
    volume_total = df_ke5z_group['Volume'].sum()
    volume_nao_nulo = df_ke5z_group['Volume'].notna().sum()
    print(f"   Total de Volume após merge: {volume_total:,.0f}")
    print(f"   Linhas com Volume não nulo: {volume_nao_nulo:,} ({volume_nao_nulo/len(df_ke5z_group)*100:.1f}%)")
    if volume_nao_nulo < len(df_ke5z_group) * 0.5:
        print(f"   ⚠️ ATENÇÃO: Menos de 50% das linhas têm Volume! Verifique o merge.")
else:
    print(f"   ⚠️ ERRO: Coluna 'Volume' não encontrada após merge!")

# Se após o merge ainda existir mais de uma coluna Volume, remove as extras mantendo apenas 'Volume'
colunas_volume = [col for col in df_ke5z_group.columns if 'Volume' in str(col) and col != 'Volume']
if colunas_volume:
    df_ke5z_group = df_ke5z_group.drop(columns=colunas_volume)
    print(f"Colunas Volume duplicadas removidas: {colunas_volume}")

# A coluna já vem como 'Custo' desde o merge na célula 1, não precisa renomear

# Mostrar o df_ke5z_group
display(df_ke5z_group)

# Somar coluna Total
print('Total = ', df_ke5z_group['Total'].sum()) 


# Gerar excel com o df_ke5z_group
arquivo_excel_group = CAMINHO_DF_KE5Z_GROUP_XLSX if 'CAMINHO_DF_KE5Z_GROUP_XLSX' in globals() else os.path.join(PASTA_BUD, 'df_ke5z_group_BUD.xlsx')
df_ke5z_group.to_excel(arquivo_excel_group, index=False)
# gerar um arquivo parquet com o df_ke5z_group
arquivo_parquet_group = CAMINHO_DF_KE5Z_GROUP if 'CAMINHO_DF_KE5Z_GROUP' in globals() else os.path.join(PASTA_BUD, 'df_ke5z_group_BUD.parquet')
df_ke5z_group.to_parquet(arquivo_parquet_group)







In [ ]:
# ====================================================================
# 💾 SALVAR RESULTADOS E CONSOLIDAR HISTÓRICO
# ====================================================================

print(f"\n{'='*70}")
print(f"💾 SALVANDO RESULTADOS DO ANO {ANO_ATUAL}")
print(f"{'='*70}")

# Verificar se as variáveis de configuração existem
if 'ANO_ATUAL' not in globals():
    print("⚠️  Variáveis de configuração não encontradas!")
    print("⚠️  Execute a primeira célula (Configuração do Ano) primeiro!")
    print("⚠️  Salvando na raiz do projeto...")
    
    # Salvar na raiz (comportamento padrão)
    df_final.to_parquet('df_final.parquet')
    df_vol.to_parquet('df_vol.parquet')
    df_ke5z_group.to_parquet('df_ke5z_group.parquet')
    print("✅ Arquivos salvos na raiz do projeto")
else:
    # ====================================================================
    # 1. Adicionar coluna de ano em todos os DataFrames
    # ====================================================================
    
    if 'Ano' not in df_final.columns:
        df_final['Ano'] = ANO_ATUAL
    if 'Ano' not in df_vol.columns:
        df_vol['Ano'] = ANO_ATUAL
    if 'Ano' not in df_ke5z_group.columns:
        df_ke5z_group['Ano'] = ANO_ATUAL
    
    # ====================================================================
    # 2. Salvar Parquets na pasta BUD
    # ====================================================================
    
    print(f"\n📄 Salvando Parquets em: {PASTA_BUD}/")
    
    df_final.to_parquet(CAMINHO_DF_FINAL)
    print(f"   ✅ df_final_BUD.parquet salvo ({len(df_final):,} linhas)")
    
    df_vol.to_parquet(CAMINHO_DF_VOL)
    print(f"   ✅ df_vol_BUD.parquet salvo ({len(df_vol):,} linhas)")
    
    df_ke5z_group.to_parquet(CAMINHO_DF_KE5Z_GROUP)
    print(f"   ✅ df_ke5z_group_BUD.parquet salvo ({len(df_ke5z_group):,} linhas)")
    
    # ====================================================================
    # 3. Salvar Excel na pasta BUD
    # ====================================================================
    
    print(f"\n📊 Salvando Excel em: {PASTA_BUD}/")
    
    df_final.to_excel(CAMINHO_DF_FINAL_XLSX, index=False)
    print(f"   ✅ df_final_BUD.xlsx salvo")
    
    df_vol.to_excel(CAMINHO_DF_VOL_XLSX, index=False)
    print(f"   ✅ df_vol_BUD.xlsx salvo")
    
    df_ke5z_group.to_excel(CAMINHO_DF_KE5Z_GROUP_XLSX, index=False)
    print(f"   ✅ df_ke5z_group_BUD.xlsx salvo")
    
    # ====================================================================
    # 4. Consolidar com Histórico
    # ====================================================================
    
    def consolidar_historico(df_novo, caminho_historico, nome_df):
        """Consolida dados de TODOS os anos disponíveis nas pastas"""
        
        # Buscar todos os anos disponíveis nas pastas
        pasta_dados = 'dados'
        anos_disponiveis = []
        
        if os.path.exists(pasta_dados):
            for item in os.listdir(pasta_dados):
                caminho_item = os.path.join(pasta_dados, item)
                if os.path.isdir(caminho_item) and item.isdigit():
                    anos_disponiveis.append(int(item))
        
        anos_disponiveis = sorted(anos_disponiveis)
        print(f"   📚 Consolidando {nome_df} de TODOS os anos disponíveis...")
        print(f"      Anos encontrados nas pastas: {anos_disponiveis}")
        
        # Carregar dados de todos os anos disponíveis (da pasta BUD)
        dfs_todos_anos = []
        for ano in anos_disponiveis:
            # Determinar qual arquivo buscar baseado no nome_df (na pasta BUD)
            pasta_ano_bud = os.path.join(pasta_dados, str(ano), 'BUD')
            if nome_df == 'df_final':
                caminho_ano = os.path.join(pasta_ano_bud, 'df_final_BUD.parquet')
            elif nome_df == 'df_vol':
                caminho_ano = os.path.join(pasta_ano_bud, 'df_vol_BUD.parquet')
            elif nome_df == 'df_ke5z_group':
                caminho_ano = os.path.join(pasta_ano_bud, 'df_ke5z_group_BUD.parquet')
            else:
                continue
            
            if os.path.exists(caminho_ano):
                try:
                    df_ano = pd.read_parquet(caminho_ano)
                    # Garantir que a coluna Ano existe
                    if 'Ano' not in df_ano.columns:
                        df_ano['Ano'] = ano
                    dfs_todos_anos.append(df_ano)
                    print(f"      ✅ Carregado {nome_df} do ano {ano} ({len(df_ano):,} registros)")
                except Exception as e:
                    print(f"      ⚠️ Erro ao carregar {caminho_ano}: {e}")
        
        if len(dfs_todos_anos) > 0:
            # Consolidar todos os anos
            df_consolidado = pd.concat(dfs_todos_anos, ignore_index=True)
            
            # Remover duplicatas se houver (baseado em todas as colunas exceto índice)
            df_consolidado = df_consolidado.drop_duplicates()
            
            # Salvar
            df_consolidado.to_parquet(caminho_historico)
            
            anos_finais = sorted(df_consolidado['Ano'].unique())
            print(f"   ✅ {nome_df} consolidado!")
            print(f"      Total de registros: {len(df_consolidado):,}")
            print(f"      Anos disponíveis: {anos_finais}")
            
            return df_consolidado
        else:
            print(f"   ⚠️ Nenhum arquivo encontrado para {nome_df}")
            # Se não encontrou nenhum arquivo, usar apenas o df_novo
            df_consolidado = df_novo
            df_consolidado.to_parquet(caminho_historico)
            return df_consolidado
    
    print(f"\n📚 Consolidando Histórico BUD em: {PASTA_HISTORICO_BUD}/")
    
    df_final_historico = consolidar_historico(df_final, CAMINHO_HISTORICO_FINAL, 'df_final')
    df_vol_historico = consolidar_historico(df_vol, CAMINHO_HISTORICO_VOL, 'df_vol')
    df_ke5z_historico = consolidar_historico(df_ke5z_group, CAMINHO_HISTORICO_KE5Z, 'df_ke5z_group')
    
    # ====================================================================
    # 5. Atualizar log de processamento
    # ====================================================================
    
    log_path = os.path.join(PASTA_BUD, '.processamento_log_BUD.txt')
    with open(log_path, 'w', encoding='utf-8') as f:
        f.write(f"Processamento de Dados BUD - Ano {ANO_ATUAL}\n")
        f.write(f"{'='*50}\n")
        f.write(f"Data/Hora: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
        f.write(f"Arquivos processados:\n")
        for arquivo in arquivos_ok:
            f.write(f"  - {arquivo}\n")
        f.write(f"\nProcessamento concluído: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
        f.write(f"Arquivos gerados (BUD):\n")
        f.write(f"  - df_final_BUD.parquet ({len(df_final):,} linhas)\n")
        f.write(f"  - df_vol_BUD.parquet ({len(df_vol):,} linhas)\n")
        f.write(f"  - df_ke5z_group_BUD.parquet ({len(df_ke5z_group):,} linhas)\n")
    
    # ====================================================================
    # 📊 RESUMO FINAL
    # ====================================================================
    
    print(f"\n{'='*70}")
    print(f"✅ PROCESSAMENTO BUD CONCLUÍDO COM SUCESSO!")
    print(f"{'='*70}")
    print(f"📅 Ano processado: {ANO_ATUAL}")
    print(f"\n📁 Arquivos salvos em (BUD):")
    print(f"   {PASTA_BUD}/")
    print(f"      ├── df_final_BUD.parquet ({len(df_final):,} linhas)")
    print(f"      ├── df_vol_BUD.parquet ({len(df_vol):,} linhas)")
    print(f"      ├── df_ke5z_group_BUD.parquet ({len(df_ke5z_group):,} linhas)")
    print(f"      ├── df_final_BUD.xlsx")
    print(f"      ├── df_vol_BUD.xlsx")
    print(f"      ├── df_ke5z_group_BUD.xlsx")
    print(f"      └── df_final_cpu_BUD.xlsx")
    print(f"\n📚 Histórico consolidado BUD em:")
    print(f"   {PASTA_HISTORICO_BUD}/")
    print(f"      ├── df_final_historico_BUD.parquet (anos: {sorted(df_final_historico['Ano'].unique())})")
    print(f"      ├── df_vol_historico_BUD.parquet (anos: {sorted(df_vol_historico['Ano'].unique())})")
    print(f"      └── df_ke5z_historico_BUD.parquet (anos: {sorted(df_ke5z_historico['Ano'].unique())})")
    print(f"{'='*70}")
    print(f"\n🎯 Próximos passos:")
    print(f"   1. Verifique os arquivos gerados em: {PASTA_BUD}/")
    print(f"   2. Execute o Streamlit para visualizar os dados")
    print(f"   3. Para processar outro ano, reinicie o notebook")
    print(f"{'='*70}\n")
